# Discovery + Silver: `billing.invoice_items`

La tabla mas grande del proyecto (150,000 filas). Chequeamos ademas que `line_total = quantity * unit_price` sea consistente.

In [1]:
import sys
sys.path.append("/home/jovyan/work/src")

import pandas as pd
from utils.db import get_engine

engine = get_engine()
df = pd.read_sql("SELECT * FROM bronze.billing__invoice_items", engine)
df.shape

(150000, 9)

## 1. Forma general

In [2]:
print(df.dtypes)
df.head()

invoice_item_id            object
quantity                   object
unit_price                 object
line_total                 object
invoice_id                 object
product_id                 object
_source_file               object
_ingested_at       datetime64[ns]
_dag_run_id                object
dtype: object


,invoice_item_id,quantity,unit_price,line_total,invoice_id,product_id,_source_file,_ingested_at,_dag_run_id
0,INI-000000001,7,64.65,452.55000000000007,INV-00029386,PRD-00117,billing/invoice_items.csv,2026-07-17 15:16:34.172462,manual__2026-07-17T15:16:30+00:00
1,INI-000000002,1,20.39,20.39,INV-00045051,PRD-00142,billing/invoice_items.csv,2026-07-17 15:16:34.172462,manual__2026-07-17T15:16:30+00:00
2,INI-000000003,7,36.59,256.13,INV-00029813,PRD-00117,billing/invoice_items.csv,2026-07-17 15:16:34.172462,manual__2026-07-17T15:16:30+00:00
3,INI-000000004,8,49.38,395.04,INV-00032937,PRD-00110,billing/invoice_items.csv,2026-07-17 15:16:34.172462,manual__2026-07-17T15:16:30+00:00
4,INI-000000005,3,24.91,74.73,INV-00032440,PRD-00140,billing/invoice_items.csv,2026-07-17 15:16:34.172462,manual__2026-07-17T15:16:30+00:00


## 2. Nulos, duplicados e integridad referencial

In [3]:
print("Nulos por columna:")
print(df.isna().sum())
print()
print("invoice_item_id duplicados:", df["invoice_item_id"].duplicated().sum())

invoices = pd.read_sql("SELECT invoice_id FROM silver.billing__invoices", engine)
products = pd.read_sql("SELECT product_id FROM silver.billing__products", engine)
print("invoice_id huerfanos:", (~df["invoice_id"].isin(invoices["invoice_id"])).sum())
print("product_id huerfanos:", (~df["product_id"].isin(products["product_id"])).sum())

Nulos por columna:
invoice_item_id    0
quantity           0
unit_price         0
line_total         0
invoice_id         0
product_id         0
_source_file       0
_ingested_at       0
_dag_run_id        0
dtype: int64

invoice_item_id duplicados: 0


invoice_id huerfanos: 0
product_id huerfanos: 0


## 3. `quantity`, `unit_price` y consistencia de `line_total`

In [4]:
quantity = pd.to_numeric(df["quantity"], errors="coerce")
unit_price = pd.to_numeric(df["unit_price"], errors="coerce")
line_total = pd.to_numeric(df["line_total"], errors="coerce")

print("quantity < 1:", (quantity < 1).sum())
print("unit_price < 0:", (unit_price < 0).sum())

esperado = (quantity * unit_price).round(2)
diff = (line_total - esperado).abs()
print("line_total inconsistente (dif > 0.01):", (diff > 0.01).sum())

quantity < 1: 0
unit_price < 0: 0
line_total inconsistente (dif > 0.01): 0


## 4. Conclusion

Tabla limpia (sin nulos, sin duplicados, 0 FKs huerfanas, `line_total` consistente con `quantity * unit_price` en el 100% de las filas). Solo tipado.

## 5. Limpieza con pandas

In [5]:
df_silver = df[["invoice_item_id", "invoice_id", "product_id", "quantity", "unit_price", "line_total"]].copy()

df_silver["quantity"] = pd.to_numeric(df_silver["quantity"], errors="raise").astype(int)
df_silver["unit_price"] = pd.to_numeric(df_silver["unit_price"], errors="raise")
df_silver["line_total"] = pd.to_numeric(df_silver["line_total"], errors="raise")

df_silver.head()

,invoice_item_id,invoice_id,product_id,quantity,unit_price,line_total
0,INI-000000001,INV-00029386,PRD-00117,7,64.65,452.55
1,INI-000000002,INV-00045051,PRD-00142,1,20.39,20.39
2,INI-000000003,INV-00029813,PRD-00117,7,36.59,256.13
3,INI-000000004,INV-00032937,PRD-00110,8,49.38,395.04
4,INI-000000005,INV-00032440,PRD-00140,3,24.91,74.73


## 6. Validar antes de escribir

In [6]:
assert len(df_silver) == len(df)
assert df_silver["invoice_item_id"].is_unique
assert df_silver["invoice_id"].isin(invoices["invoice_id"]).all()
assert df_silver["product_id"].isin(products["product_id"]).all()
assert df_silver["quantity"].ge(1).all()
print("OK:", len(df_silver), "filas listas para silver")

OK: 150000 filas listas para silver


## 7. Escribir en `silver.billing__invoice_items`

In [7]:
df_silver["_silver_loaded_at"] = pd.Timestamp.utcnow()

df_silver.to_sql(
    "billing__invoice_items",
    engine,
    schema="silver",
    if_exists="replace",
    index=False,
    method="multi",
    chunksize=5000,
)
print("Escrito en silver.billing__invoice_items")

Escrito en silver.billing__invoice_items


## 8. Verificar

In [8]:
check = pd.read_sql("SELECT * FROM silver.billing__invoice_items LIMIT 5", engine)
print(pd.read_sql("SELECT count(*) AS filas, count(DISTINCT invoice_item_id) AS ids_unicos FROM silver.billing__invoice_items", engine))
check

    filas  ids_unicos
0  150000      150000


,invoice_item_id,invoice_id,product_id,quantity,unit_price,line_total,_silver_loaded_at
0,INI-000000001,INV-00029386,PRD-00117,7,64.65,452.55,2026-07-17 15:17:51.806340+00:00
1,INI-000000002,INV-00045051,PRD-00142,1,20.39,20.39,2026-07-17 15:17:51.806340+00:00
2,INI-000000003,INV-00029813,PRD-00117,7,36.59,256.13,2026-07-17 15:17:51.806340+00:00
3,INI-000000004,INV-00032937,PRD-00110,8,49.38,395.04,2026-07-17 15:17:51.806340+00:00
4,INI-000000005,INV-00032440,PRD-00140,3,24.91,74.73,2026-07-17 15:17:51.806340+00:00
